In [ ]:
# Устанавливаем необходимые библиотеки
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_filename)  # db_filename уже определена
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Все таблицы:")
print(tables['name'].tolist())

for table in tables['name']:
    cols = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print(f"\nТаблица {table}: {cols['name'].tolist()}")

conn.close()


import sqlite3
import pandas as pd

conn = sqlite3.connect(db_filename)

tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables = tables_df['name'].tolist()
print("Таблицы в базе данных:")
print(tables)

columns_info = {}
for table in tables:
    cols = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    columns_info[table] = cols['name'].tolist()
    print(f"\nТаблица '{table}': {columns_info[table]}")

# Простой уровень
print("\n=== ПРОСТОЙ УРОВЕНЬ ===")
owner_table = None
for table in tables:
    if 'owner' in table.lower():
        owner_table = table
        break

if owner_table is None:
    print("Не найдена таблица с владельцами. Завершение.")
    conn.close()
    exit()

if 'last_name' in columns_info[owner_table]:
    query_simple = f'SELECT * FROM {owner_table} WHERE last_name LIKE "%ова";'
    owners_ova = pd.read_sql_query(query_simple, conn)
    print(f"Найдено владельцев с фамилией на 'ова': {len(owners_ova)}")
    display(owners_ova)
else:
    print(f"В таблице {owner_table} нет поля last_name. Поля: {columns_info[owner_table]}")

# Продвинутый уровень
print("\n=== ПРОДВИНУТЫЙ УРОВЕНЬ ===")

# Проверяем наличие нужных таблиц
if 'owners' in tables and 'wings' in tables:

    query_advanced = """
    SELECT
        o.id,
        o.first_name,
        o.last_name,
        o.birth_date,
        COUNT(w.id) AS wings_count
    FROM owners o
    JOIN wings w ON o.id = w.owner_id
    GROUP BY o.id
    ORDER BY wings_count DESC, o.birth_date DESC
    LIMIT 5;
    """

    result = pd.read_sql_query(query_advanced, conn)

    print("Топ-5 самых молодых владельцев с максимальным числом экспонатов:")
    display(result)

else:
    print("Не найдены необходимые таблицы (owners, wings)")

# Продвинутый уровень 2.0 геомаркетинг

print("\n=== Продвинутый уровень, ГЕОМАРКЕТИНГ ===")

query_geo = """
SELECT
    SUBSTR(p.location, INSTR(p.location, ',') + 2) AS city,
    SUM(m.price) AS total_spent,
    AVG(m.price) AS avg_price
FROM moves m
JOIN places p ON m.place_id = p.id
GROUP BY city
ORDER BY total_spent DESC;
"""

geo_result = pd.read_sql_query(query_geo, conn)

print("Самые перспективные города для рекламы:")
display(geo_result)


Все таблицы:
['owners', 'types', 'places', 'wings', 'moves']

Таблица owners: ['id', 'email', 'first_name', 'last_name', 'middle_name', 'birth_date']

Таблица types: ['id', 'name']

Таблица places: ['id', 'location', 'scale']

Таблица wings: ['id', 'owner_id', 'type_id', 'profit', 'name']

Таблица moves: ['id', 'wing_id', 'place_id', 'price', 'dt']
Таблицы в базе данных:
['owners', 'types', 'places', 'wings', 'moves']

Таблица 'owners': ['id', 'email', 'first_name', 'last_name', 'middle_name', 'birth_date']

Таблица 'types': ['id', 'name']

Таблица 'places': ['id', 'location', 'scale']

Таблица 'wings': ['id', 'owner_id', 'type_id', 'profit', 'name']

Таблица 'moves': ['id', 'wing_id', 'place_id', 'price', 'dt']

=== ПРОСТОЙ УРОВЕНЬ ===
Найдено владельцев с фамилией на 'ова': 46


,id,email,first_name,last_name,middle_name,birth_date
0,7,gbogdanova@example.org,Людмила,Кузнецова,Ильинична,2001-03-12
1,8,radovan_1990@example.org,Алена,Семенова,None,2007-10-17
2,14,upetrova@example.com,Яна,Макарова,Антоновна,1968-12-23
3,18,timofe68@example.com,Виктория,Богданова,Юрьевна,1996-08-09
4,20,svjatoslavsilin@example.net,Диана,Макарова,Петровна,1960-08-01
5,22,petuhovaevfrosinija@example.org,Алиса,Семенова,Валентиновна,2002-02-01
6,23,simonsolovev@example.net,Вероника,Комарова,Дмитриевна,2010-08-07
7,26,tatjanapanfilova@example.net,Вера,Орлова,Федоровна,1971-07-16
8,28,ljubosmisl_20@example.com,Екатерина,Иванова,Григорьевна,1984-09-13
9,33,qnikiforov@example.com,Ангелина,Смирнова,Даниловна,1999-12-24



=== ПРОДВИНУТЫЙ УРОВЕНЬ ===
Топ-5 самых молодых владельцев с максимальным числом экспонатов:


,id,first_name,last_name,birth_date,wings_count
0,84,Владимир,Воробьев,2008-07-10,60
1,58,Елена,Беляева,2008-03-17,60
2,144,Оксана,Орлова,2002-04-12,60
3,42,Григорий,Смирнов,2001-05-22,60
4,102,Александр,Богданов,1994-12-20,60



=== ГЕОМАРКЕТИНГ ===
Самые перспективные города для рекламы:


,city,total_spent,avg_price
0,"пр. Маяковского, д. 416 стр. 8/7",1.305451e+08,25860.762542
1,"ул. Вишневая, д. 67 к. 8/5",1.289663e+08,25782.954960
2,"пер. Геологов, д. 1/7",1.288623e+08,25618.748175
3,"бул. Восточный, д. 832 к. 33",1.286390e+08,25774.195903
4,"алл. Приморская, д. 1/2 к. 246",1.284163e+08,25911.284903
5,"ш. Народное, д. 831 к. 280",1.283096e+08,25718.498326
6,"наб. Кузнецова, д. 3/2 стр. 535",1.282176e+08,25731.001064
7,"пер. Новгородский, д. 31 стр. 53",1.280651e+08,25434.965533
8,"пр. Театральный, д. 59 стр. 10",1.277501e+08,25458.369063
9,"наб. Шаумяна, д. 7",1.277025e+08,25267.600607
